# 3D Reconstruction Comparison: VGGT vs Depth Anything 3

Compare **VGGT-1B** and **Depth Anything 3 (DA3)** multi-view 3D reconstruction on the same input images.

Both models independently predict depth maps, confidence, camera poses, and 3D point clouds — making this a true end-to-end comparison.

**Pipeline:**
1. Load images from Google Drive
2. Run VGGT-1B reconstruction (point cloud + depth + poses)
3. Run DA3 reconstruction (point cloud + depth + poses)
4. Generate multi-angle visualizations for both
5. Comparative analysis (side-by-side renders, depth maps, quantitative metrics)
6. Export all results timestamped to Google Drive
7. Interactive viser viewer for the best result

**Requirements:** Google Colab with A100 GPU

In [1]:
# ============================================================
# Configuration — edit these before running
# ============================================================

INPUT_IMAGES_PATH = "/content/drive/MyDrive/COMPVIS/capture_data/images"
OUTPUT_BASE_PATH  = "/content/drive/MyDrive/3DVisionData/LeRobot"

MAX_FRAMES     = 20
CONF_THRESHOLD = 0.3
VOXEL_RESOLUTION = 64
VISER_PORT     = 8081

In [2]:
%%capture install_output
import subprocess, sys, os

def run(cmd):
    subprocess.check_call(cmd, shell=True)

# VGGT
if not os.path.exists("/content/vggt"):
    run("git clone https://github.com/facebookresearch/vggt.git /content/vggt")
run("pip install -e /content/vggt")

# Depth Anything 3
if not os.path.exists("/content/Depth-Anything-3"):
    run("git clone --recursive https://github.com/ByteDance-Seed/Depth-Anything-3.git /content/Depth-Anything-3")
run("pip install -r /content/Depth-Anything-3/requirements.txt")

# Other dependencies
run("pip install viser matplotlib opencv-python-headless trimesh numba Pillow scipy")

sys.path.insert(0, "/content/Depth-Anything-3/src")
print("All dependencies installed.")

In [3]:
import sys, os
sys.path.insert(0, "/content/Depth-Anything-3/src")

from google.colab import drive
from pathlib import Path
from datetime import datetime

drive.mount("/content/drive")

img_dir = Path(INPUT_IMAGES_PATH)
assert img_dir.exists(), f"Input path not found: {img_dir}\nCreate a shortcut in MyDrive to the shared folder."

exts = {".jpg", ".jpeg", ".png", ".bmp"}
image_paths = sorted(p for p in img_dir.iterdir() if p.suffix.lower() in exts)
print(f"Found {len(image_paths)} images in {img_dir}")
for p in image_paths[:5]:
    print(f"  {p.name}")
if len(image_paths) > 5:
    print(f"  ... and {len(image_paths) - 5} more")

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_dir = Path(OUTPUT_BASE_PATH) / timestamp
output_dir.mkdir(parents=True, exist_ok=True)
print(f"\nOutput directory: {output_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 56 images in /content/drive/MyDrive/COMPVIS/capture_data/images
  frame_00000.jpg
  frame_00001.jpg
  frame_00002.jpg
  frame_00003.jpg
  frame_00004.jpg
  ... and 51 more

Output directory: /content/drive/MyDrive/3DVisionData/LeRobot/2026-03-17_16-40-39


In [4]:
import cv2
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

def load_frames(folder: Path, max_frames: int) -> tuple[list[np.ndarray], list[Path]]:
    """Load images sorted by filename, return (bgr_frames, paths)."""
    exts = {".jpg", ".jpeg", ".png", ".bmp"}
    paths = sorted(p for p in folder.iterdir() if p.suffix.lower() in exts)
    paths = paths[:max_frames]
    frames, used_paths = [], []
    for p in paths:
        img = cv2.imread(str(p))
        if img is None:
            print(f"  WARNING: could not read {p.name}, skipping")
            continue
        frames.append(img)
        used_paths.append(p)
        print(f"  Loaded {p.name}  ({img.shape[1]}x{img.shape[0]})")
    return frames, used_paths

frames, frame_paths = load_frames(Path(INPUT_IMAGES_PATH), MAX_FRAMES)
print(f"\n{len(frames)} frame(s) loaded")

# Display thumbnail grid
n = len(frames)
cols = min(n, 5)
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
if rows == 1 and cols == 1:
    axes = np.array([axes])
axes = np.atleast_2d(axes)
for i in range(rows * cols):
    ax = axes[i // cols][i % cols]
    if i < n:
        ax.imshow(cv2.cvtColor(frames[i], cv2.COLOR_BGR2RGB))
        ax.set_title(frame_paths[i].name, fontsize=7)
    ax.axis("off")
plt.suptitle(f"Input Images ({n} frames)", fontweight="bold")
plt.tight_layout()
plt.show()

  Loaded frame_00000.jpg  (640x320)
  Loaded frame_00001.jpg  (640x320)
  Loaded frame_00002.jpg  (640x320)
  Loaded frame_00003.jpg  (640x320)
  Loaded frame_00004.jpg  (640x320)
  Loaded frame_00005.jpg  (640x320)
  Loaded frame_00006.jpg  (640x320)
  Loaded frame_00007.jpg  (640x320)
  Loaded frame_00008.jpg  (640x320)
  Loaded frame_00009.jpg  (640x320)
  Loaded frame_00010.jpg  (640x320)
  Loaded frame_00011.jpg  (640x320)
  Loaded frame_00012.jpg  (640x320)
  Loaded frame_00013.jpg  (640x320)
  Loaded frame_00014.jpg  (640x320)
  Loaded frame_00015.jpg  (640x320)
  Loaded frame_00016.jpg  (640x320)
  Loaded frame_00017.jpg  (640x320)
  Loaded frame_00018.jpg  (640x320)
  Loaded frame_00019.jpg  (640x320)

20 frame(s) loaded


In [13]:
import time
import torch
from vggt.models.vggt import VGGT
from vggt.utils.pose_enc import pose_encoding_to_extri_intri

device = "cuda" if torch.cuda.is_available() else "cpu"
cap = torch.cuda.get_device_capability() if device == "cuda" else (0, 0)
dtype = torch.bfloat16 if cap[0] >= 8 else torch.float16

print(f"Device: {device}  dtype: {dtype}")
print("Loading VGGT-1B ...")
t0 = time.time()
vggt_model = VGGT.from_pretrained("facebook/VGGT-1B").to(device).eval()
print(f"VGGT loaded in {time.time() - t0:.1f}s")

# Prepare images: BGR -> RGB, resize width=518, height divisible by 14
TARGET_W = 518
processed = []
for bgr in frames:
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    h, w = rgb.shape[:2]
    new_h = int(round(h * TARGET_W / w / 14)) * 14
    resized = cv2.resize(rgb, (TARGET_W, new_h), interpolation=cv2.INTER_AREA)
    tensor = torch.from_numpy(resized).permute(2, 0, 1).float() / 255.0
    processed.append(tensor)

max_h = max(t.shape[1] for t in processed)
padded = []
for t in processed:
    if t.shape[1] < max_h:
        pad = torch.zeros(3, max_h - t.shape[1], TARGET_W)
        t = torch.cat([t, pad], dim=1)
    padded.append(t)

images_tensor = torch.stack(padded).to(device)  # (S, 3, H, W)
S, _, H, W = images_tensor.shape
print(f"Input tensor: {images_tensor.shape}")

# Forward pass
print("Running VGGT forward pass ...")
t0 = time.perf_counter()
with torch.no_grad(), torch.amp.autocast("cuda", dtype=dtype):
    preds = vggt_model(images_tensor)

extrinsic, intrinsic = pose_encoding_to_extri_intri(preds["pose_enc"], (H, W))
preds["extrinsic"] = extrinsic
preds["intrinsic"] = intrinsic

np_preds = {}
for k, v in preds.items():
    if isinstance(v, torch.Tensor):
        np_preds[k] = v.cpu().float().numpy().squeeze(0)

vggt_depth = np_preds["depth"].squeeze(-1) if np_preds["depth"].ndim == 4 else np_preds["depth"]
vggt_depth_conf = np_preds["depth_conf"]
vggt_world_pts = np_preds["world_points"]
vggt_world_conf = np_preds["world_points_conf"]
vggt_extrinsics = np_preds["extrinsic"]
vggt_intrinsics = np_preds["intrinsic"]

# Fuse point cloud with confidence filtering
images_np = images_tensor.cpu().numpy()
all_xyz, all_rgb, all_conf = [], [], []
for s in range(S):
    conf = vggt_world_conf[s]
    mask = conf > CONF_THRESHOLD
    pts = vggt_world_pts[s][mask]
    rgb = (images_np[s].transpose(1, 2, 0) * 255).astype(np.uint8)
    colours = rgb[mask]
    all_xyz.append(pts)
    all_rgb.append(colours)
    all_conf.append(conf[mask])

vggt_fused_xyz = np.concatenate(all_xyz) if all_xyz else np.zeros((0, 3))
vggt_fused_rgb = np.concatenate(all_rgb) if all_rgb else np.zeros((0, 3), dtype=np.uint8)
vggt_fused_conf = np.concatenate(all_conf) if all_conf else np.zeros((0,))
vggt_time = time.perf_counter() - t0

# Free GPU memory for DA3
del vggt_model, images_tensor, preds
torch.cuda.empty_cache()

print(f"\nVGGT Reconstruction Summary")
print(f"{'='*45}")
print(f"  Frames processed : {S}")
print(f"  Processing time  : {vggt_time:.1f}s")
print(f"  Fused points     : {len(vggt_fused_xyz):,}")
print(f"  Mean confidence  : {vggt_fused_conf.mean():.3f}" if len(vggt_fused_conf) > 0 else "  Mean confidence  : N/A")
if len(vggt_fused_xyz) > 0:
    mn = vggt_fused_xyz.min(axis=0)
    mx = vggt_fused_xyz.max(axis=0)
    print(f"  Bounds (m)       : [{mn[0]:.2f},{mn[1]:.2f},{mn[2]:.2f}] -> [{mx[0]:.2f},{mx[1]:.2f},{mx[2]:.2f}]")
print(f"{'='*45}")

Device: cuda  dtype: torch.bfloat16
Loading VGGT-1B ...
VGGT loaded in 12.3s
Input tensor: torch.Size([20, 3, 252, 518])
Running VGGT forward pass ...


  with torch.cuda.amp.autocast(enabled=False):




VGGT Reconstruction Summary
  Frames processed : 20
  Processing time  : 0.8s
  Fused points     : 2,610,720
  Mean confidence  : 1.000
  Bounds (m)       : [-0.35,-1.12,0.16] -> [1.24,1.06,1.35]


In [23]:
import sys, time, torch
import numpy as np
sys.path.insert(0, "/content/Depth-Anything-3/src")
from depth_anything_3.api import DepthAnything3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading DA3 model ...")
t0 = time.time()
da3_model = DepthAnything3.from_pretrained("depth-anything/DA3NESTED-GIANT-LARGE")
da3_model = da3_model.to(device=device)
print(f"DA3 loaded in {time.time() - t0:.1f}s")

# DA3 takes file paths as input
da3_image_paths = sorted([str(p) for p in frame_paths])

print(f"Running DA3 inference on {len(da3_image_paths)} images ...")
t0 = time.perf_counter()
da3_prediction = da3_model.inference(da3_image_paths, infer_gs=True)
da3_time = time.perf_counter() - t0
print(f"DA3 inference done in {da3_time:.1f}s")

def _to_np(x):
    if hasattr(x, 'cpu'):
        return x.detach().cpu().numpy()
    return np.asarray(x)

da3_depth = _to_np(da3_prediction.depth)           # (N, H, W)
da3_conf = _to_np(da3_prediction.conf)             # (N, H, W)
da3_extrinsics = _to_np(da3_prediction.extrinsics) # (N, 3, 4)
da3_intrinsics = _to_np(da3_prediction.intrinsics) # (N, 3, 3)
da3_images_np = _to_np(da3_prediction.processed_images)  # (N, H, W, 3) uint8

print(f"  Depth shape      : {da3_depth.shape}")
print(f"  Extrinsics shape : {da3_extrinsics.shape}")
print(f"  Intrinsics shape : {da3_intrinsics.shape}")

# Build DA3 point cloud from native 3D Gaussian means (already in world space)
# rather than manual depth unprojection which is sensitive to extrinsic conventions.
N_da3, H_da3, W_da3 = da3_depth.shape
da3_conf_flat = da3_conf.reshape(-1)
da3_rgb_flat = da3_images_np.reshape(-1, 3)

da3_gaussian_xyz = None
if hasattr(da3_prediction, "gaussians") and da3_prediction.gaussians is not None:
    gs = da3_prediction.gaussians.means
    print(f"  3D Gaussian means: {gs.shape}")
    da3_gaussian_xyz = _to_np(gs).squeeze()

if da3_gaussian_xyz is not None and len(da3_gaussian_xyz) == len(da3_conf_flat):
    print("  Using Gaussian means as DA3 point cloud (native world-space output)")
    mask = da3_conf_flat > CONF_THRESHOLD
    da3_fused_xyz = da3_gaussian_xyz[mask].astype(np.float64)
    da3_fused_rgb = da3_rgb_flat[mask]
    da3_fused_conf = da3_conf_flat[mask]
else:
    print("  Gaussian means unavailable — falling back to depth unprojection")
    da3_all_xyz, da3_all_rgb, da3_all_conf = [], [], []
    for s in range(N_da3):
        depth_s = da3_depth[s]
        conf_s = da3_conf[s]
        K = da3_intrinsics[s]
        E = da3_extrinsics[s]
        R = E[:3, :3]
        t = E[:3, 3]

        fx, fy = K[0, 0], K[1, 1]
        cx, cy = K[0, 2], K[1, 2]

        mask = conf_s > CONF_THRESHOLD
        vs, us = np.where(mask)
        d = depth_s[mask]

        x_cam = (us - cx) * d / fx
        y_cam = (vs - cy) * d / fy
        z_cam = d
        cam_pts = np.stack([x_cam, y_cam, z_cam], axis=-1)

        world_pts = (R @ cam_pts.T + t[:, None]).T

        rgb_s = da3_images_np[s]
        colours = rgb_s[mask]

        da3_all_xyz.append(world_pts)
        da3_all_rgb.append(colours)
        da3_all_conf.append(conf_s[mask])

    da3_fused_xyz = np.concatenate(da3_all_xyz) if da3_all_xyz else np.zeros((0, 3))
    da3_fused_rgb = np.concatenate(da3_all_rgb) if da3_all_rgb else np.zeros((0, 3), dtype=np.uint8)
    da3_fused_conf = np.concatenate(da3_all_conf) if da3_all_conf else np.zeros((0,))

del da3_model
torch.cuda.empty_cache()

print(f"\nDA3 Reconstruction Summary")
print(f"{'='*45}")
print(f"  Frames processed : {N_da3}")
print(f"  Processing time  : {da3_time:.1f}s")
print(f"  Fused points     : {len(da3_fused_xyz):,}")
print(f"  Mean confidence  : {da3_fused_conf.mean():.3f}" if len(da3_fused_conf) > 0 else "  Mean confidence  : N/A")
if len(da3_fused_xyz) > 0:
    mn = da3_fused_xyz.min(axis=0)
    mx = da3_fused_xyz.max(axis=0)
    print(f"  Bounds (m)       : [{mn[0]:.2f},{mn[1]:.2f},{mn[2]:.2f}] -> [{mx[0]:.2f},{mx[1]:.2f},{mx[2]:.2f}]")
print(f"{'='*45}")

Loading DA3 model ...
[INFO ] using SwiGLU layer as FFN
[INFO ] using MLP layer as FFN
DA3 loaded in 16.0s
Running DA3 inference on 20 images ...
[INFO ] Processed Images Done taking 0.08166766166687012 seconds. Shape:  torch.Size([20, 3, 252, 504])
[INFO ] Selecting reference view using strategy: saddle_balanced
[INFO ] Model Forward Pass Done. Time: 0.8554141521453857 seconds
[INFO ] Conversion to Prediction Done. Time: 0.013241052627563477 seconds
DA3 inference done in 1.0s
  Depth shape      : (20, 252, 504)
  Extrinsics shape : (20, 3, 4)
  Intrinsics shape : (20, 3, 3)
  3D Gaussian means: torch.Size([1, 2540160, 3])
  Using Gaussian means as DA3 point cloud (native world-space output)

DA3 Reconstruction Summary
  Frames processed : 20
  Processing time  : 1.0s
  Fused points     : 2,540,160
  Mean confidence  : 1.440
  Bounds (m)       : [-125.87,-182.03,-43.15] -> [103.35,230.88,277.15]


In [24]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from pathlib import Path
import numpy as np

VIEWPOINTS = [
    ("front",      0,   0),
    ("back",       0, 180),
    ("left",       0,  90),
    ("right",      0, 270),
    ("top",       90,   0),
    ("isometric", 30,  45),
]

def render_pointcloud_multiangle(
    xyz: np.ndarray,
    rgb: np.ndarray,
    title: str,
    save_dir: Path,
    max_points: int = 50_000,
) -> list[Path]:
    """Render a point cloud from 6 fixed viewpoints, save PNGs, return paths."""
    save_dir.mkdir(parents=True, exist_ok=True)

    if len(xyz) > max_points:
        idx = np.random.default_rng(42).choice(len(xyz), max_points, replace=False)
        xyz_s, rgb_s = xyz[idx], rgb[idx]
    else:
        xyz_s, rgb_s = xyz, rgb

    rgb_norm = rgb_s.astype(float) / 255.0 if rgb_s.max() > 1 else rgb_s.astype(float)

    saved = []
    fig, axes = plt.subplots(2, 3, figsize=(18, 12), subplot_kw={"projection": "3d"})
    for i, (name, elev, azim) in enumerate(VIEWPOINTS):
        ax = axes[i // 3][i % 3]
        ax.scatter(xyz_s[:, 0], xyz_s[:, 1], xyz_s[:, 2],
                   c=rgb_norm, s=0.3, alpha=0.6)
        ax.view_init(elev=elev, azim=azim)
        ax.set_title(f"{name} (elev={elev}, azim={azim})", fontsize=9)
        ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")

        # Also save individual view
        fig_single = plt.figure(figsize=(8, 8))
        ax_s = fig_single.add_subplot(111, projection="3d")
        ax_s.scatter(xyz_s[:, 0], xyz_s[:, 1], xyz_s[:, 2],
                     c=rgb_norm, s=0.5, alpha=0.7)
        ax_s.view_init(elev=elev, azim=azim)
        ax_s.set_title(f"{title} — {name}", fontweight="bold")
        ax_s.set_xlabel("X"); ax_s.set_ylabel("Y"); ax_s.set_zlabel("Z")
        single_path = save_dir / f"{name}.png"
        fig_single.savefig(str(single_path), dpi=120, bbox_inches="tight")
        plt.close(fig_single)
        saved.append(single_path)

    fig.suptitle(f"{title} — Multi-Angle Views ({len(xyz):,} points)", fontsize=14, fontweight="bold")
    plt.tight_layout()
    grid_path = save_dir / "grid.png"
    fig.savefig(str(grid_path), dpi=120, bbox_inches="tight")
    saved.append(grid_path)
    plt.show()
    plt.close(fig)
    return saved

print("Visualization utility defined.")

Visualization utility defined.


In [25]:
vggt_render_dir = output_dir / "vggt" / "renders"
vggt_saved = render_pointcloud_multiangle(
    vggt_fused_xyz, vggt_fused_rgb,
    title="VGGT-1B",
    save_dir=vggt_render_dir,
)
print(f"VGGT renders saved to {vggt_render_dir}  ({len(vggt_saved)} files)")

VGGT renders saved to /content/drive/MyDrive/3DVisionData/LeRobot/2026-03-17_16-40-39/vggt/renders  (7 files)


In [26]:
da3_render_dir = output_dir / "da3" / "renders"
da3_saved = render_pointcloud_multiangle(
    da3_fused_xyz, da3_fused_rgb,
    title="Depth Anything 3",
    save_dir=da3_render_dir,
)
print(f"DA3 renders saved to {da3_render_dir}  ({len(da3_saved)} files)")

DA3 renders saved to /content/drive/MyDrive/3DVisionData/LeRobot/2026-03-17_16-40-39/da3/renders  (7 files)


In [27]:
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa
from scipy.stats import pearsonr
import numpy as np
from pathlib import Path

comp_dir = output_dir / "comparison"
comp_dir.mkdir(parents=True, exist_ok=True)

# ================================================================
# 1. Side-by-side multi-angle renders (VGGT top row, DA3 bottom)
# ================================================================
max_pts = 40_000
rng = np.random.default_rng(42)

def subsample(xyz, rgb, n):
    if len(xyz) <= n:
        return xyz, rgb
    idx = rng.choice(len(xyz), n, replace=False)
    return xyz[idx], rgb[idx]

v_xyz, v_rgb = subsample(vggt_fused_xyz, vggt_fused_rgb, max_pts)
d_xyz, d_rgb = subsample(da3_fused_xyz, da3_fused_rgb, max_pts)
v_rgb_n = v_rgb.astype(float) / 255.0 if v_rgb.max() > 1 else v_rgb.astype(float)
d_rgb_n = d_rgb.astype(float) / 255.0 if d_rgb.max() > 1 else d_rgb.astype(float)

fig, axes = plt.subplots(2, 6, figsize=(30, 10), subplot_kw={"projection": "3d"})
for i, (name, elev, azim) in enumerate(VIEWPOINTS):
    for row, (xyz, rgb_n, label) in enumerate([(v_xyz, v_rgb_n, "VGGT"), (d_xyz, d_rgb_n, "DA3")]):
        ax = axes[row][i]
        ax.scatter(xyz[:, 0], xyz[:, 1], xyz[:, 2], c=rgb_n, s=0.3, alpha=0.6)
        ax.view_init(elev=elev, azim=azim)
        ax.set_title(f"{label} — {name}", fontsize=8)
        ax.set_xlabel("X", fontsize=6); ax.set_ylabel("Y", fontsize=6); ax.set_zlabel("Z", fontsize=6)
fig.suptitle("Side-by-Side: VGGT (top) vs DA3 (bottom)", fontsize=14, fontweight="bold")
plt.tight_layout()
sbs_path = comp_dir / "side_by_side.png"
fig.savefig(str(sbs_path), dpi=120, bbox_inches="tight")
plt.show(); plt.close(fig)
print(f"Side-by-side saved: {sbs_path}")

# ================================================================
# 2. Per-frame depth map comparison
# ================================================================
n_compare = min(len(vggt_depth), N_da3, 4)
fig, axes = plt.subplots(n_compare, 4, figsize=(20, 5 * n_compare))
if n_compare == 1:
    axes = axes[np.newaxis, :]

depth_correlations = []
for s in range(n_compare):
    vd = vggt_depth[s]
    dd = da3_depth[s]
    # Resize to match if needed
    if vd.shape != dd.shape:
        dd_resized = cv2.resize(dd, (vd.shape[1], vd.shape[0]), interpolation=cv2.INTER_LINEAR)
    else:
        dd_resized = dd

    vmax = max(np.percentile(vd, 98), np.percentile(dd_resized, 98))
    diff = np.abs(vd - dd_resized)

    axes[s, 0].imshow(vd, cmap="viridis", vmin=0, vmax=vmax)
    axes[s, 0].set_title(f"VGGT Depth (frame {s})", fontsize=9)
    axes[s, 1].imshow(dd_resized, cmap="viridis", vmin=0, vmax=vmax)
    axes[s, 1].set_title(f"DA3 Depth (frame {s})", fontsize=9)
    im = axes[s, 2].imshow(diff, cmap="hot")
    axes[s, 2].set_title(f"|Difference|", fontsize=9)
    plt.colorbar(im, ax=axes[s, 2], fraction=0.046)

    # Depth correlation on valid pixels
    valid = (vd > 0) & (dd_resized > 0)
    if valid.sum() > 100:
        r, _ = pearsonr(vd[valid].ravel(), dd_resized[valid].ravel())
        depth_correlations.append(r)
        axes[s, 3].scatter(vd[valid].ravel()[::50], dd_resized[valid].ravel()[::50],
                          s=1, alpha=0.3)
        axes[s, 3].set_xlabel("VGGT depth"); axes[s, 3].set_ylabel("DA3 depth")
        axes[s, 3].set_title(f"Correlation r={r:.3f}", fontsize=9)
        axes[s, 3].plot([0, vmax], [0, vmax], "r--", alpha=0.5)
    for ax in axes[s]:
        ax.axis("on")

fig.suptitle("Depth Map Comparison: VGGT vs DA3", fontsize=14, fontweight="bold")
plt.tight_layout()
depth_path = comp_dir / "depth_comparison.png"
fig.savefig(str(depth_path), dpi=120, bbox_inches="tight")
plt.show(); plt.close(fig)
print(f"Depth comparison saved: {depth_path}")

# ================================================================
# 3. Camera pose comparison
# ================================================================
def cam_positions_from_extrinsics(extr):
    """Extract world-space camera positions from (S, 3, 4) extrinsics."""
    R = extr[:, :3, :3]
    t = extr[:, :3, 3]
    return -np.einsum("sij,sj->si", R.transpose(0, 2, 1), t)

vggt_cam_pos = cam_positions_from_extrinsics(vggt_extrinsics)
da3_cam_pos = cam_positions_from_extrinsics(da3_extrinsics)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(*vggt_cam_pos.T, c="blue", s=60, marker="^", label="VGGT cameras", zorder=5)
ax.scatter(*da3_cam_pos.T, c="red", s=60, marker="v", label="DA3 cameras", zorder=5)
for i in range(len(vggt_cam_pos)):
    ax.plot(*zip(vggt_cam_pos[i], da3_cam_pos[i]), "k--", alpha=0.3, linewidth=0.5)
ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
ax.set_title("Camera Pose Comparison", fontweight="bold")
ax.legend()
plt.tight_layout()
pose_path = comp_dir / "pose_comparison.png"
fig.savefig(str(pose_path), dpi=120, bbox_inches="tight")
plt.show(); plt.close(fig)
print(f"Pose comparison saved: {pose_path}")

# ================================================================
# 4. Quantitative metrics
# ================================================================
def bbox_volume(xyz):
    if len(xyz) == 0:
        return 0.0
    mn, mx = xyz.min(axis=0), xyz.max(axis=0)
    ext = mx - mn
    return float(np.prod(ext))

metrics = {
    "vggt": {
        "point_count": int(len(vggt_fused_xyz)),
        "processing_time_s": round(vggt_time, 2),
        "mean_confidence": round(float(vggt_fused_conf.mean()), 4) if len(vggt_fused_conf) > 0 else None,
        "std_confidence": round(float(vggt_fused_conf.std()), 4) if len(vggt_fused_conf) > 0 else None,
        "bbox_volume_m3": round(bbox_volume(vggt_fused_xyz), 6),
        "mean_depth": round(float(vggt_depth.mean()), 4),
        "std_depth": round(float(vggt_depth.std()), 4),
    },
    "da3": {
        "point_count": int(len(da3_fused_xyz)),
        "processing_time_s": round(da3_time, 2),
        "mean_confidence": round(float(da3_fused_conf.mean()), 4) if len(da3_fused_conf) > 0 else None,
        "std_confidence": round(float(da3_fused_conf.std()), 4) if len(da3_fused_conf) > 0 else None,
        "bbox_volume_m3": round(bbox_volume(da3_fused_xyz), 6),
        "mean_depth": round(float(da3_depth.mean()), 4),
        "std_depth": round(float(da3_depth.std()), 4),
    },
    "comparison": {
        "depth_correlations": [round(float(r), 4) for r in depth_correlations],
        "mean_depth_correlation": round(float(np.mean(depth_correlations)), 4) if depth_correlations else None,
        "mean_camera_position_diff_m": round(float(np.linalg.norm(vggt_cam_pos - da3_cam_pos, axis=1).mean()), 4)
            if len(vggt_cam_pos) == len(da3_cam_pos) else None,
    },
}

# Determine winner
vggt_score, da3_score = 0, 0
if metrics["vggt"]["point_count"] > metrics["da3"]["point_count"]:
    vggt_score += 1
else:
    da3_score += 1
if (metrics["vggt"]["mean_confidence"] or 0) > (metrics["da3"]["mean_confidence"] or 0):
    vggt_score += 1
else:
    da3_score += 1
if metrics["vggt"]["processing_time_s"] < metrics["da3"]["processing_time_s"]:
    vggt_score += 1
else:
    da3_score += 1

winner = "vggt" if vggt_score > da3_score else "da3"
metrics["winner"] = winner

print("\n" + "=" * 60)
print("  QUANTITATIVE COMPARISON")
print("=" * 60)
for method in ["vggt", "da3"]:
    tag = " << WINNER" if method == winner else ""
    print(f"\n  {method.upper()}{tag}")
    for k, v in metrics[method].items():
        print(f"    {k:30s}: {v}")
print(f"\n  Depth correlations: {metrics['comparison']['depth_correlations']}")
print(f"  Mean depth correlation: {metrics['comparison']['mean_depth_correlation']}")
print(f"  Mean camera pos diff : {metrics['comparison']['mean_camera_position_diff_m']} m")
print(f"\n  Winner: {winner.upper()}")
print("=" * 60)

Side-by-side saved: /content/drive/MyDrive/3DVisionData/LeRobot/2026-03-17_16-40-39/comparison/side_by_side.png
Depth comparison saved: /content/drive/MyDrive/3DVisionData/LeRobot/2026-03-17_16-40-39/comparison/depth_comparison.png
Pose comparison saved: /content/drive/MyDrive/3DVisionData/LeRobot/2026-03-17_16-40-39/comparison/pose_comparison.png

  QUANTITATIVE COMPARISON

  VGGT << WINNER
    point_count                   : 2610720
    processing_time_s             : 0.78
    mean_confidence               : 1.0
    std_confidence                : 0.0005
    bbox_volume_m3                : 4.122715
    mean_depth                    : 0.8943
    std_depth                     : 0.256

  DA3
    point_count                   : 2540160
    processing_time_s             : 1.02
    mean_confidence               : 1.4397
    std_confidence                : 0.5528
    bbox_volume_m3                : 30316253.686712
    mean_depth                    : 0.8254
    std_depth                     

In [28]:
import json, shutil
from pathlib import Path
import numpy as np

# ── VGGT artifacts ────────────────────────────────────────────
vggt_dir = output_dir / "vggt"
vggt_dir.mkdir(parents=True, exist_ok=True)
np.savez_compressed(str(vggt_dir / "point_cloud.npz"),
                    xyz=vggt_fused_xyz, rgb=vggt_fused_rgb, conf=vggt_fused_conf,
                    extrinsics=vggt_extrinsics, intrinsics=vggt_intrinsics)
np.savez_compressed(str(vggt_dir / "depth_maps.npz"),
                    depth=vggt_depth, depth_conf=vggt_depth_conf)
print(f"VGGT point cloud : {vggt_dir / 'point_cloud.npz'}")
print(f"VGGT depth maps  : {vggt_dir / 'depth_maps.npz'}")

# ── DA3 artifacts ─────────────────────────────────────────────
da3_dir = output_dir / "da3"
da3_dir.mkdir(parents=True, exist_ok=True)
da3_pc_data = dict(xyz=da3_fused_xyz, rgb=da3_fused_rgb, conf=da3_fused_conf,
                   extrinsics=da3_extrinsics, intrinsics=da3_intrinsics)
if da3_gaussian_xyz is not None:
    da3_pc_data["gaussian_means"] = da3_gaussian_xyz
np.savez_compressed(str(da3_dir / "point_cloud.npz"), **da3_pc_data)
np.savez_compressed(str(da3_dir / "depth_maps.npz"),
                    depth=da3_depth, conf=da3_conf)
print(f"DA3 point cloud  : {da3_dir / 'point_cloud.npz'}")
print(f"DA3 depth maps   : {da3_dir / 'depth_maps.npz'}")

# ── Comparison artifacts ──────────────────────────────────────
with open(str(comp_dir / "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics JSON     : {comp_dir / 'metrics.json'}")

# ── Copy input images ─────────────────────────────────────────
img_out = output_dir / "input_images"
img_out.mkdir(parents=True, exist_ok=True)
for p in frame_paths:
    shutil.copy2(str(p), str(img_out / p.name))
print(f"Input images     : {img_out}  ({len(frame_paths)} files)")

# ── Summary ───────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"  ALL RESULTS EXPORTED")
print(f"{'='*60}")
print(f"  Timestamp : {timestamp}")
print(f"  Output dir: {output_dir}")

total_size = 0
for f in output_dir.rglob("*"):
    if f.is_file():
        total_size += f.stat().st_size
print(f"  Total size: {total_size / 1024 / 1024:.1f} MB")
print(f"  Winner    : {metrics['winner'].upper()}")
print(f"{'='*60}")

VGGT point cloud : /content/drive/MyDrive/3DVisionData/LeRobot/2026-03-17_16-40-39/vggt/point_cloud.npz
VGGT depth maps  : /content/drive/MyDrive/3DVisionData/LeRobot/2026-03-17_16-40-39/vggt/depth_maps.npz
DA3 point cloud  : /content/drive/MyDrive/3DVisionData/LeRobot/2026-03-17_16-40-39/da3/point_cloud.npz
DA3 depth maps   : /content/drive/MyDrive/3DVisionData/LeRobot/2026-03-17_16-40-39/da3/depth_maps.npz
Metrics JSON     : /content/drive/MyDrive/3DVisionData/LeRobot/2026-03-17_16-40-39/comparison/metrics.json
Input images     : /content/drive/MyDrive/3DVisionData/LeRobot/2026-03-17_16-40-39/input_images  (20 files)

  ALL RESULTS EXPORTED
  Timestamp : 2026-03-17_16-40-39
  Output dir: /content/drive/MyDrive/3DVisionData/LeRobot/2026-03-17_16-40-39
  Total size: 125.5 MB
  Winner    : VGGT


In [40]:
# Align DA3 to VGGT coordinate frame via camera-pose Procrustes
import numpy as np

def umeyama(src, dst):
    """Closed-form similarity transform: dst ≈ s*R@src + t  (Umeyama 1991)."""
    n = src.shape[0]
    mu_s, mu_d = src.mean(0), dst.mean(0)
    sc, dc = src - mu_s, dst - mu_d
    var_s = np.sum(sc ** 2) / n
    H = dc.T @ sc / n
    U, S, Vt = np.linalg.svd(H)
    d = np.sign(np.linalg.det(U) * np.linalg.det(Vt))
    D = np.diag([1.0, 1.0, d])
    R = U @ D @ Vt
    s = np.trace(np.diag(S) @ D) / var_s
    t = mu_d - s * R @ mu_s
    return s, R, t

def cam_centers_w2c(extr):
    R, t = extr[:, :3, :3], extr[:, :3, 3]
    return -np.einsum("sij,sj->si", R.transpose(0, 2, 1), t)

def cam_centers_c2w(extr):
    return extr[:, :3, 3].copy()

# VGGT extrinsics are world-to-camera
vggt_cams = cam_centers_w2c(vggt_extrinsics)

# Try both conventions for DA3 and pick the one with lower residual
best_res = np.inf
for label, da3_cams in [("w2c", cam_centers_w2c(da3_extrinsics)),
                        ("c2w", cam_centers_c2w(da3_extrinsics))]:
    s, R, t = umeyama(da3_cams, vggt_cams)
    aligned = s * (R @ da3_cams.T).T + t
    res = np.linalg.norm(aligned - vggt_cams, axis=1).mean()
    print(f"  Convention {label}: scale={s:.6f}  residual={res:.6f}")
    if res < best_res:
        best_res, best_s, best_R, best_t, best_label = res, s, R, t, label

# Apply the winning transform to the full DA3 point cloud
da3_aligned_xyz = (best_s * (best_R @ da3_fused_xyz.astype(np.float64).T).T + best_t).astype(np.float32)
da3_aligned_rgb = da3_fused_rgb
da3_aligned_conf = da3_fused_conf

print(f"\nUsing {best_label} convention  (scale={best_s:.6f}, residual={best_res:.6f})")
print(f"  DA3 original : {da3_fused_xyz.min(0).round(2)} → {da3_fused_xyz.max(0).round(2)}")
print(f"  DA3 aligned  : {da3_aligned_xyz.min(0).round(2)} → {da3_aligned_xyz.max(0).round(2)}")
print(f"  VGGT         : {vggt_fused_xyz.min(0).round(2)} → {vggt_fused_xyz.max(0).round(2)}")

# Save
da3_aligned_dir = output_dir / "da3_aligned"
da3_aligned_dir.mkdir(parents=True, exist_ok=True)
np.savez_compressed(str(da3_aligned_dir / "point_cloud.npz"),
                    xyz=da3_aligned_xyz, rgb=da3_aligned_rgb, conf=da3_aligned_conf)
print(f"  Saved: {da3_aligned_dir / 'point_cloud.npz'}")

  Convention w2c: scale=1.139925  residual=0.100336
  Convention c2w: scale=0.673231  residual=0.271142

Using w2c convention  (scale=1.139925, residual=0.100336)
  DA3 original : [-125.87 -182.03  -43.15] → [103.35 230.88 277.15]
  DA3 aligned  : [-391.13 -206.58  -51.48] → [ 98.38 135.83 231.3 ]
  VGGT         : [-0.35 -1.12  0.16] → [1.24 1.06 1.35]
  Saved: /content/drive/MyDrive/3DVisionData/LeRobot/2026-03-17_16-40-39/da3_aligned/point_cloud.npz


In [41]:
import viser
import viser.transforms as vtf
import time
import numpy as np
from google.colab import output as colab_output

# ── Shut down any previous server ────────────────────────────
if "server" in dir() and server is not None:
    try:
        server.close()
        time.sleep(1)
    except Exception:
        pass

MAX_VISER_POINTS = 800_000

def subsample(xyz, rgb, conf, max_pts):
    """Subsample arrays to max_pts, keeping highest-confidence points."""
    if len(xyz) <= max_pts:
        return xyz, rgb, conf
    top_idx = np.argpartition(conf, -max_pts)[-max_pts:]
    return xyz[top_idx], rgb[top_idx], conf[top_idx]

# ── Prepare per-model data (subsampled for viser) ────────────
models = {}
for label, xyz, rgb, conf, extr, intr in [
    ("VGGT", vggt_fused_xyz,  vggt_fused_rgb,   vggt_fused_conf,  vggt_extrinsics, vggt_intrinsics),
    ("DA3",  da3_aligned_xyz, da3_aligned_rgb,   da3_aligned_conf, da3_extrinsics,  da3_intrinsics),
]:
    s_xyz, s_rgb, s_conf = subsample(
        xyz.astype(np.float32),
        rgb.astype(np.uint8),
        conf.astype(np.float32),
        MAX_VISER_POINTS,
    )
    models[label] = dict(xyz=s_xyz, rgb=s_rgb, conf=s_conf, extr=extr, intr=intr,
                         full_count=len(xyz))
    print(f"  {label}: {len(xyz):,} pts -> {len(s_xyz):,} for viewer")

global_center = np.mean(
    np.concatenate([m["xyz"] for m in models.values()]), axis=0
)

def invert_extrinsics(ext):
    c2w = np.zeros_like(ext)
    R = ext[:, :3, :3]
    t = ext[:, :3, 3]
    c2w[:, :3, :3] = R.transpose(0, 2, 1)
    c2w[:, :3, 3] = -(R.transpose(0, 2, 1) @ t[:, :, None])[:, :, 0]
    return c2w

# ── Server ────────────────────────────────────────────────────
server = viser.ViserServer(host="0.0.0.0", port=VISER_PORT)
server.gui.configure_theme(titlebar_content=None, control_layout="collapsible")

# ── GUI controls ──────────────────────────────────────────────
gui_model = server.gui.add_dropdown(
    "Model", options=["Both", "VGGT", "DA3"], initial_value="Both")
gui_status = server.gui.add_text("Status", initial_value="")
gui_conf = server.gui.add_slider(
    "Confidence threshold %", min=0, max=100, step=1, initial_value=10)
gui_point_size = server.gui.add_slider(
    "Point size", min=0.0005, max=0.01, step=0.0005, initial_value=0.002)
gui_show_cameras = server.gui.add_checkbox("Show cameras", initial_value=True)

# ── Scene elements (one point cloud + cameras per model) ──────
pc_handles = {}
cam_frame_handles = {}

for key, m in models.items():
    pc_handles[key] = server.scene.add_point_cloud(
        f"pc/{key}",
        points=m["xyz"] - global_center,
        colors=m["rgb"],
        point_size=0.002,
        point_shape="circle",
    )

    c2w = invert_extrinsics(m["extr"])
    cam_frame_handles[key] = []
    for i in range(len(m["extr"])):
        R = c2w[i, :3, :3]
        t = c2w[i, :3, 3]
        wxyz = vtf.SO3.from_matrix(R).wxyz
        frame = server.scene.add_frame(
            f"cams/{key}/cam_{i}",
            wxyz=wxyz, position=t - global_center,
            axes_length=0.03, axes_radius=0.001, origin_radius=0.001,
        )
        fy = m["intr"][i, 1, 1]
        H = int(round(m["intr"][i, 1, 2] * 2))
        W = int(round(m["intr"][i, 0, 2] * 2))
        fov = float(2 * np.arctan2(H / 2, fy)) if fy > 0 else 0.7
        asp = W / H if H > 0 else 1.0
        server.scene.add_camera_frustum(
            f"cams/{key}/cam_{i}/frustum",
            fov=fov, aspect=asp, scale=0.04, line_width=1.0,
        )
        cam_frame_handles[key].append(frame)

# ── Sync helpers ──────────────────────────────────────────────
def refresh_visibility():
    sel = gui_model.value
    show_cams = gui_show_cameras.value
    for key in models:
        active = sel in (key, "Both")
        pc_handles[key].visible = active
        for h in cam_frame_handles[key]:
            h.visible = active and show_cams

    parts = []
    for key in models:
        if sel in (key, "Both"):
            m = models[key]
            parts.append(f"{key}: {len(m['xyz']):,}/{m['full_count']:,} pts")
    gui_status.value = " | ".join(parts)

def refresh_points():
    sel = gui_model.value
    thr_pct = gui_conf.value
    ps = gui_point_size.value
    for key, m in models.items():
        if sel not in (key, "Both"):
            continue
        thr = np.percentile(m["conf"], thr_pct) if len(m["conf"]) > 0 else 0
        mask = m["conf"] >= thr
        pc_handles[key].points = m["xyz"][mask] - global_center
        pc_handles[key].colors = m["rgb"][mask]
        pc_handles[key].point_size = ps

refresh_visibility()
refresh_points()

# ── Callbacks ─────────────────────────────────────────────────
@gui_model.on_update
def _(_):
    refresh_visibility()
    refresh_points()

@gui_conf.on_update
def _(_):
    refresh_points()

@gui_point_size.on_update
def _(_):
    for key in models:
        pc_handles[key].point_size = gui_point_size.value

@gui_show_cameras.on_update
def _(_):
    refresh_visibility()

# ── Share & embed ─────────────────────────────────────────────
try:
    share_url = server.request_share_url()
    print(f"\nShareable viser link: {share_url}")
except Exception as e:
    print(f"(Share URL not available: {e})")

print(f"\nInteractive 3D viewer — use dropdown to switch: VGGT / DA3 / Both")
print(f"VGGT: {models['VGGT']['full_count']:,} pts  |  DA3: {models['DA3']['full_count']:,} pts")
print(f"(Viewer shows up to {MAX_VISER_POINTS:,} highest-confidence points per model)")
colab_output.serve_kernel_port_as_iframe(VISER_PORT, height=600)

  VGGT: 2,610,720 pts -> 800,000 for viewer
  DA3: 2,540,160 pts -> 800,000 for viewer


╭────── viser (listening *:8093) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8093   │
│   Websocket │ ws://localhost:8093     │
│             ╵                         │
╰───────────────────────────────────────╯

(viser) Share URL requested!

(viser) Generated share URL (expires in 24 hours, max 16 clients): https://imu-digital.share.viser.studio


Shareable viser link: https://imu-digital.share.viser.studio

Interactive 3D viewer — use dropdown to switch: VGGT / DA3 / Both
VGGT: 2,610,720 pts  |  DA3: 2,540,160 pts
(Viewer shows up to 800,000 highest-confidence points per model)


<IPython.core.display.Javascript object>

(viser) Connection closed (0, 0 total)

(viser) Connection opened (0, 1 total), 267 persistent messages